# Study 01: LangSmith 트레이싱 학습

**목표**: LangSmith를 이해하고 LLM 앱 모니터링 방법을 배웁니다.

**소요 시간**: 약 30분

**비용: 무료** (로컬 Ollama + LangSmith 무료 플랜)

---

## 1. LangSmith란?

### 한 줄 요약
**LangSmith = LLM 앱 전용 모니터링/디버깅 플랫폼**

### 비유로 이해하기

| 웹 개발 | LLM 개발 |
|---------|----------|
| 크롬 개발자 도구 | **LangSmith** |
| Network 탭에서 API 호출 확인 | 트레이스에서 LLM 호출 확인 |
| Console에서 에러 확인 | 입력/출력 확인 |

### 트레이싱(Tracing)이란?
LLM을 호출할 때마다 다음을 자동 기록:
- **입력**: 프롬프트, 시스템 메시지
- **출력**: LLM 응답
- **메타데이터**: 지연 시간, 모델명

```
[사용자 질문] → [프롬프트 구성] → [LLM 호출] → [응답]
     ↓              ↓              ↓           ↓
  (기록)         (기록)         (기록)      (기록)
```

## 2. 왜 필요한가?

### 문제 상황
```
"LLM이 이상한 답변을 했는데... 왜지?"
"프롬프트가 어떻게 전달됐지?"
"응답이 왜 이렇게 느리지?"
```

### LangSmith로 해결
```
모든 LLM 호출 기록 → 문제 원인 추적 가능
프롬프트 전문 확인 → 디버깅 쉬움
지연 시간 측정 → 성능 최적화
```

### 현업 표준
- **프로덕션 LLM 앱 = 반드시 모니터링 필요**
- LangSmith 외에도: Langfuse, Weights & Biases 등
- LangChain 생태계 → LangSmith가 가장 쉬움

## 3. 환경 설정

### Step 1: LangSmith 가입
1. https://smith.langchain.com 접속
2. 회원가입 (무료 - 월 5000 트레이스)
3. 로그인

### Step 2: API 키 발급
1. 좌측 하단 Settings 클릭
2. "API Keys" 탭 선택
3. "Create API Key" 클릭
4. 키 복사 (ls_... 형태)

### Step 3: .env 파일에 저장 (현업 표준)

**절대 코드에 직접 API 키를 넣지 마세요!**

프로젝트 루트의 `.env` 파일에 추가:
```bash
# .env 파일에 추가
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=ls_여기에_본인_키_입력
```

**왜 .env를 사용하나요?**
- 노트북이 Git에 올라가도 키 노출 방지
- `.gitignore`에 `.env` 추가되어 있음

In [1]:
# 필요한 패키지 설치
!pip install -q langsmith langchain-ollama python-dotenv

In [7]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드 (현업 표준)
load_dotenv()

# 설정 확인
langsmith_key = os.environ.get("LANGSMITH_API_KEY", "")

print("환경 변수 로드 완료!")
print(f"  LANGSMITH_TRACING: {os.environ.get('LANGSMITH_TRACING', 'not set')}")
if langsmith_key:
    print(f"  LANGSMITH_API_KEY: {langsmith_key[-4:]}...")
else:
    print("  LANGSMITH_API_KEY 없음 - .env 파일 확인!")

환경 변수 로드 완료!
  LANGSMITH_TRACING: true
  LANGSMITH_API_KEY: 05cf...


### Ollama 서버 확인

이 프로젝트는 **로컬 LLM (Ollama)**을 사용합니다. 비용이 들지 않습니다!

Ollama가 실행 중이어야 합니다:
```bash
# 터미널에서 Ollama 시작
ollama serve
```

In [8]:
import requests

# Ollama 서버 상태 확인
try:
    response = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = response.json().get("models", [])
    print("Ollama 서버 연결 성공!")
    print(f"사용 가능한 모델: {[m['name'] for m in models]}")
except:
    print("Ollama 서버에 연결할 수 없습니다.")
    print("터미널에서 'ollama serve' 실행 후 다시 시도하세요.")

Ollama 서버 연결 성공!
사용 가능한 모델: ['qwen3-hr:latest', 'snowflake-arctic-embed2:latest', 'dengcao/Qwen3-Embedding-8B:Q5_K_M', 'qwen3:8b', 'mistral:7b', 'llama3.1:8b']


## 4. 기본 트레이싱: LangChain + Ollama

### 핵심 개념
- LangChain 사용 시 **자동으로 LangSmith 트레이싱**
- 환경 변수만 설정하면 끝!
- 코드 변경 없이 모든 LLM 호출 기록

```python
# 그냥 LangChain 사용하면 자동 트레이싱!
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen3-hr:latest")
```

In [9]:
from langchain_ollama import ChatOllama

# Ollama LLM 클라이언트 생성 (자동으로 LangSmith에 트레이싱!)
llm = ChatOllama(
    model="qwen3-hr:latest",  # 프로젝트에서 사용하는 모델
    base_url="http://localhost:11434",
    temperature=0
)

print("ChatOllama 클라이언트 생성 완료!")
print("LangSmith 트레이싱이 자동으로 활성화됩니다.")

ChatOllama 클라이언트 생성 완료!
LangSmith 트레이싱이 자동으로 활성화됩니다.


In [10]:
# LLM 호출 (자동으로 LangSmith에 기록됨!)
response = llm.invoke("안녕하세요! LangSmith 테스트입니다. 짧게 연결완료라고 답변해주세요.")

print("응답:", response.content)
print()
print("이제 https://smith.langchain.com 에서 트레이스를 확인해보세요!")

응답: SELECT '연결완료' as status FROM dual

이제 https://smith.langchain.com 에서 트레이스를 확인해보세요!


### 확인해보기
1. https://smith.langchain.com 접속
2. 상단의 프로젝트 드롭다운에서 "default" (또는 설정한 프로젝트명) 선택
3. 트레이스 목록에서 방금 실행한 항목 클릭
4. (선택) 우상단 "Dashboard" 버튼으로 대시보드 확인

**확인할 수 있는 정보:**
- 입력 메시지
- 출력 응답
- 응답 시간

## 5. @traceable 데코레이터

### 핵심 개념
- `@traceable`: 함수 전체를 하나의 트레이스로 기록
- 여러 LLM 호출을 **계층 구조**로 묶을 수 있음
- 함수 이름, 입력, 출력 자동 기록

```python
@traceable(name="my_pipeline")
def my_function(input):
    # LLM 호출 1
    # LLM 호출 2
    return result
```

**LangSmith UI에서 보이는 구조:**
```
my_pipeline
  └── ChatOllama (호출 1)
  └── ChatOllama (호출 2)
```

In [11]:
from langsmith import traceable

@traceable(name="hr_qa_pipeline")
def ask_hr_question(question: str) -> str:
    """HR 관련 질문에 답변하는 파이프라인"""
    
    # 시스템 프롬프트 + 사용자 질문
    messages = [
        ("system", "당신은 HR 전문가입니다. 간단히 답변해주세요."),
        ("user", question)
    ]
    
    # LLM 호출 (자동으로 하위 트레이스로 기록)
    response = llm.invoke(messages)
    
    return response.content

print("@traceable 데코레이터가 적용된 함수 생성 완료!")

@traceable 데코레이터가 적용된 함수 생성 완료!


In [12]:
# 함수 실행 → LangSmith에서 계층 구조로 확인 가능
result = ask_hr_question("연차 신청은 어떻게 하나요?")

print("답변:")
print(result)
print()
print("LangSmith에서 'hr_qa_pipeline' 트레이스를 확인해보세요!")

답변:
연차는 월 1일씩 부여되며, 월 1일 이내 사용이 원칙입니다. 연차 사용 전 인사팀 승인이 필요합니다.

LangSmith에서 'hr_qa_pipeline' 트레이스를 확인해보세요!


### 중첩 트레이스 예제

여러 단계로 구성된 파이프라인도 계층 구조로 기록됩니다.

In [13]:
@traceable(name="step1_classify")
def classify_question(question: str) -> str:
    """질문 유형 분류"""
    messages = [
        ("system", "질문 유형을 한 단어로 분류하세요: HR정책, 급여, 휴가, 기타"),
        ("user", question)
    ]
    response = llm.invoke(messages)
    return response.content

@traceable(name="step2_answer")
def generate_answer(question: str, category: str) -> str:
    """분류된 유형에 맞는 답변 생성"""
    messages = [
        ("system", f"당신은 {category} 전문가입니다. 간단히 답변하세요."),
        ("user", question)
    ]
    response = llm.invoke(messages)
    return response.content

@traceable(name="full_hr_pipeline")
def full_pipeline(question: str) -> dict:
    """전체 HR 질의응답 파이프라인"""
    # Step 1: 분류
    category = classify_question(question)
    
    # Step 2: 답변 생성
    answer = generate_answer(question, category)
    
    return {
        "question": question,
        "category": category,
        "answer": answer
    }

print("중첩 트레이스 파이프라인 생성 완료!")

중첩 트레이스 파이프라인 생성 완료!


In [14]:
# 전체 파이프라인 실행
result = full_pipeline("이번 달 급여 명세서는 어디서 확인하나요?")

print("결과:")
print(f"  질문: {result['question']}")
print(f"  분류: {result['category']}")
print(f"  답변: {result['answer'][:200]}..." if len(result['answer']) > 200 else f"  답변: {result['answer']}")
print()
print("LangSmith에서 계층 구조를 확인해보세요!")
print("  full_hr_pipeline")
print("    └── step1_classify")
print("        └── ChatOllama")
print("    └── step2_answer")
print("        └── ChatOllama")

결과:
  질문: 이번 달 급여 명세서는 어디서 확인하나요?
  분류: SELECT e.name, s.base_salary, s.date FROM salaries s JOIN employees e ON s.emp_id = e.emp_id WHERE s.date = CURRENT_DATE
  답변: SELECT e.name, s.base_salary, s.date FROM salaries s JOIN employees e ON s.emp_id = e.emp_id WHERE s.date = CURRENT_DATE

LangSmith에서 계층 구조를 확인해보세요!
  full_hr_pipeline
    └── step1_classify
        └── ChatOllama
    └── step2_answer
        └── ChatOllama


## 6. 프로젝트 적용 가이드

### enterprise-hr-agent에 적용하는 방법

#### Step 1: .env 파일 수정
```bash
# .env 파일에 추가
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=ls_...
```

#### Step 2: 에이전트 함수에 @traceable 추가
```python
# core/agents/sql_agent.py
from langsmith import traceable

@traceable(name="sql_agent")
def run_sql_agent(question: str) -> str:
    # 기존 로직 그대로
    ...

# core/agents/hr_agent.py
@traceable(name="rag_agent")
def run_rag_agent(question: str) -> str:
    # 기존 로직 그대로
    ...
```

### 적용 시 이점
- SQL Agent: 생성된 SQL 쿼리 확인, 오류 디버깅
- RAG Agent: 검색된 문서, 프롬프트 확인
- 전체: 응답 시간 모니터링

## 7. LangSmith UI 사용법

### 대시보드 구조

```
좌측 사이드바 (2025 최신)
├── Tracing (트레이스 목록)
├── Monitor (대시보드 - Prebuilt/Custom)
├── Datasets (평가 데이터셋)
├── Experiments (실험 결과)
├── Prompts (프롬프트 관리)
└── Settings (설정)
```

### 트레이스 상세 보기

1. 상단 프로젝트 드롭다운에서 프로젝트 선택
2. **Tracing** 탭에서 트레이스 목록 확인
3. 트레이스 클릭하여 상세 정보 확인
4. **Monitor** 탭에서 대시보드 확인 (메트릭, 차트)

**확인 가능한 정보:**
- **Input**: 입력 메시지/프롬프트
- **Output**: LLM 응답
- **Latency**: 응답 시간
- **Child Runs**: 하위 호출 (중첩 트레이스)

### 유용한 기능

| 기능 | 설명 |
|------|------|
| 필터링 | 날짜, 모델, 상태별 필터 |
| 검색 | 입력/출력 내용 검색 |
| 비교 | 여러 트레이스 비교 |
| 피드백 | 트레이스에 평가 추가 |

## 8. 다음 단계

### 학습 완료 체크리스트

- [ ] smith.langchain.com 가입 완료
- [ ] API 키를 .env 파일에 저장
- [ ] LangChain + Ollama 기본 트레이싱 확인
- [ ] @traceable로 커스텀 함수 트레이싱 확인
- [ ] LangSmith UI에서 트레이스 확인

### 다음 학습

**step_01_langsmith.ipynb**: 프로젝트에 실제 적용

- SQL Agent에 트레이싱 적용
- RAG Agent에 트레이싱 적용
- 실제 쿼리 디버깅 실습

---

### 참고 자료

- [LangSmith 공식 문서](https://docs.langchain.com/langsmith)
- [LangSmith Python SDK](https://github.com/langchain-ai/langsmith-sdk)

---

**수고하셨습니다!**

이제 LangSmith를 사용해서 LLM 앱을 모니터링하고 디버깅할 수 있습니다.

**비용: 무료** (로컬 Ollama + LangSmith 무료 플랜 월 5000 트레이스)